In [379]:
import numpy as np
import matplotlib.pyplot as plt
import h5py

In [380]:
def onehot_encoding(Y):
    
    num_classes = len(np.unique(Y))
    one_hot_Y = np.eye(num_classes)[Y].T

    return one_hot_Y

In [381]:
def layer_sizes(num_fc_units, layers):
    
    layer_units = []
   
    for i in range(len(layers)):
        
        if layers[i] == "Conv":
            layer_units.append("_")
            
        elif layers[i] == "FC_output":
            layer_units.append(num_fc_units[i])
            
        elif layers[i] == "FC_hidden":
            layer_units.append(num_fc_units[i])
    
    return layer_units

In [382]:
def initialize_parameters(layers, f_size, num_channels, num_filters, layer_units, conv_output):
    
    parameters = {}
    V = {}
    S = {}
    
    for l in range(1, len(layers)+1):
        
        if layers[l-1] == "Conv":
            
            parameters[f"W{l}"] = np.random.randn(f_size, f_size, num_channels, num_filters) * np.sqrt(2 / f_size)
            parameters[f"b{l}"] = np.random.randn(1, 1, 1, num_filters) * np.sqrt(2 / f_size)
    
            
            V[f"vdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            V[f"vdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

            S[f"sdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            S[f"sdb{l}"] = np.zeros(parameters[f"b{l}"].shape)
            
        elif layers[l-1] == "FC_output":
            
            parameters[f"W{l}"] = np.random.randn(layer_units[l-1], layer_units[l-2]) * np.sqrt(2 / layer_units[l-2])
            parameters[f"b{l}"] = np.zeros((layer_units[l-1], 1))

            V[f"vdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            V[f"vdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

            S[f"sdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            S[f"sdb{l}"] = np.zeros(parameters[f"b{l}"].shape)
            
        elif layers[l-1] == "FC_hidden" and layers[l-2] == "FC_hidden":
            
            parameters[f"W{l}"] = np.random.randn(layer_units[l-1], layer_units[l-2]) * np.sqrt(2 / layer_units[l-2])
            parameters[f"b{l}"] = np.zeros((layer_units[l-1], 1))

            V[f"vdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            V[f"vdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

            S[f"sdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            S[f"sdb{l}"] = np.zeros(parameters[f"b{l}"].shape)
            
        elif layers[l-1] == "FC_hidden" and layers[l-2] == "Conv":
            
            parameters[f"W{l}"] = np.random.randn(layer_units[l-1], conv_output) * np.sqrt(2 / conv_output)
            parameters[f"b{l}"] = np.zeros((layer_units[l-1], 1))

            V[f"vdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            V[f"vdb{l}"] = np.zeros(parameters[f"b{l}"].shape)

            S[f"sdW{l}"] = np.zeros(parameters[f"W{l}"].shape)
            S[f"sdb{l}"] = np.zeros(parameters[f"b{l}"].shape)
            

            
            
    return (parameters, V, S)   

In [383]:
def zero_pad(X, pad, mode, constant_values):
  
    X_pad = np.pad(X, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode=mode, constant_values=constant_values)    
    
    return X_pad

In [384]:
def relu(Z):
    
    A = np.maximum(0, Z)
    cache = Z
    
    return A, cache

In [385]:
def backward_relu(dA, Z):
    
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
      
    return dZ

In [386]:
def softmax(Z):
    
    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True) + 1e-15)
    activation_cache = Z

    return A, activation_cache  

In [387]:
def backward_softmax(dAL, Z):
    
    A = np.divide(np.exp(Z), np.sum(np.exp(Z), axis=0, keepdims=True) + 1e-15)
    dZ = A * (1 - A) * dAL
    
    return dZ

In [388]:
def convolution_forward(A_prev, W, b, hparameters):
   
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape
    stride = hparameters["stride"]
    pad = hparameters["pad"]
    pad_mode = hparameters["pad_mode"]
    constant_values = hparameters["constant_values"]
    n_H = int(((n_H_prev - f + (2 * pad)) / stride) + 1)
    n_W = int(((n_W_prev - f + (2 * pad)) / stride) + 1)
    linear_cache = (A_prev, W, b, hparameters)

    A_prev_pad = zero_pad(X=A_prev, pad=pad, mode=pad_mode, constant_values=constant_values)

    A = np.zeros((m, n_H, n_W, n_C))
    activation_cache = np.zeros((m, n_H, n_W, n_C))

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        
        for h in range(n_H):
            vert_start = h * stride
            vert_end = vert_start + f
            
            for w in range(n_W):
                horiz_start = w * stride
                horiz_end = horiz_start + f

                a_slice_prev = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                
                A[i, h, w, :] = np.sum(a_slice_prev * W, axis=(0, 1, 2)) + b
                A[i, h, w, :], activation_cache[i, h, w, :] = relu(Z=A[i, h, w, :])

    caches = (linear_cache, activation_cache)

    return A, caches

In [389]:
def fully_connected_forward(A_prev, W, b, layer):
    
    linear_cache = (A_prev, W, b)
    
    if layer == "Output":
            
        Z = np.dot(W, A_prev) + b
        A, activation_cache = softmax(Z=Z)
        
    elif layer == "Hidden":
        
        Z = np.dot(W, A_prev) + b
        A, activation_cache = relu(Z=Z)
        
    caches = (linear_cache, activation_cache)
            
    return A, caches

In [390]:
def pool_forward(A_prev, hparameters, mode):
   
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

    f = hparameters["f_pool"]
    stride = hparameters["stride"]

   
    n_H = int(1 + (n_H_prev - f) / stride)
    n_W = int(1 + (n_W_prev - f) / stride)
    n_C = n_C_prev

    A = np.zeros((m, n_H, n_W, n_C))

    for i in range(m):
        for h in range(n_H):
            
            vert_start = h * stride
            vert_end = vert_start + f
            
            for w in range(n_W):
                
                horiz_start = w * stride
                horiz_end = horiz_start + f
                a_slice_prev = A_prev[i, vert_start:vert_end, horiz_start:horiz_end, :]

                if mode == "max":
                    
                    A[i, h, w, :] = np.max(a_slice_prev, axis=(0, 1))
                    
                elif mode == "average":
                    
                    A[i, h, w, :] = np.mean(a_slice_prev, axis=(0, 1))

    cache = (A_prev, hparameters)

    return A, cache

In [391]:
def forward_propagation(X, parameters, hyperparameters, layers, pool_mode):
    
    caches = []
    A = X
    
    for l in range(1, len(layers)+1):
        
        if layers[l-1] == "Conv":
            
            W = parameters[f"W{l}"]
            b = parameters[f"b{l}"]
            A, conv_cache = convolution_forward(
                A_prev=A,
                W=W, 
                b=b, 
                hparameters=hyperparameters
            )
            
            A, pool_cache = pool_forward(
                A_prev=A, 
                hparameters=hyperparameters, 
                mode=pool_mode
            )
            caches.append((conv_cache, pool_cache))
            
        elif layers[l-1] == "FC_hidden" and layers[l-2] == "Conv":
            
            W = parameters[f"W{l}"]
            b = parameters[f"b{l}"]
            m = A.shape[0]
            A = A.reshape(m, -1).T
            A, hfc_cache = fully_connected_forward(
                A_prev=A, 
                W=W, 
                b=b, 
                layer="Hidden"
            )
            caches.append(hfc_cache)
            
        elif layers[l-1] == "FC_hidden" and layers[l-2] == "FC_hidden":
            
            W = parameters[f"W{l}"]
            b = parameters[f"b{l}"]
            
            A, hfc_cache = fully_connected_forward(
                A_prev=A, 
                W=W, 
                b=b, 
                layer="Hidden"
            )
            caches.append(hfc_cache)
            
        elif layers[l-1] == "FC_output":
            
            W = parameters[f"W{l}"]
            b = parameters[f"b{l}"]

            A, ofc_cache = fully_connected_forward(
                A_prev=A, 
                W=W, 
                b=b, 
                layer="Output"
            )
            caches.append(ofc_cache)
            
    return A, caches           

In [392]:
def compute_cost(A, Y, dataset):
    
    if dataset == "train":
        
        A = np.clip(A, 1e-15, 1 - 1e-15)
        cost = - np.sum(Y * np.log(A))
    
    else:
        
        A = np.clip(A, 1e-15, 1 - 1e-15)
        cost = - np.sum(Y * np.log(A))
        
    return cost

In [410]:
def convolution_backward(dA, cache):

    (linear_cache, activation_cache) = cache
    (A_prev, W, b, hparameters) = linear_cache
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    (f, f, n_C_prev, n_C) = W.shape

    stride = hparameters["stride"]
    pad = hparameters["pad"]
    pad_mode = hparameters["pad_mode"]
    constant_values = hparameters["constant_values"]
    
    dZ = backward_relu(dA=dA, Z=activation_cache)
    
    (m, n_H, n_W, n_C) = dZ.shape
    dA_prev = np.zeros(A_prev.shape)                          
    dW = np.zeros(W.shape)
    db = np.zeros(b.shape)
    
    A_prev_pad = zero_pad(X=A_prev, pad=pad, mode=pad_mode, constant_values=constant_values)
    dA_prev_pad = zero_pad(X=dA_prev, pad=pad, mode=pad_mode, constant_values=constant_values)
    
    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        da_prev_pad = dA_prev_pad[i]
        
        for h in range(n_H):
            for w in range(n_W):               
                for c in range(n_C):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f
                    
                    a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                    da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:,:,:,c] * dZ[i, h, w, c]
                    dW[:,:,:,c] += a_slice * dZ[i, h, w, c]
                    db[:,:,:,c] += dZ[i, h, w, c]
    
    dA_prev = dA_prev_pad[:, pad:-pad, pad:-pad, :]
    
    return dA_prev, dW, db

In [431]:
def pool_backward(dA, cache, mode, layer):
    """
    Implements the backward pass of the pooling layer.
    
    Arguments:
    dA -- gradient of the cost with respect to the output of the pooling layer, same shape as A
    cache -- cache output from the forward pass of the pooling layer, contains A_prev and hparameters
    mode -- the pooling mode, either "max" or "average"
    layer -- indicates the layer type, either "FC" for fully connected or "Conv" for convolutional
    
    Returns:
    dA_prev -- gradient of the cost with respect to the input of the pooling layer, same shape as A_prev
    """
    A_prev, hparameters = cache
    stride = hparameters["stride"]
    f = hparameters["f_pool"]
        
    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape
    
    if layer == "FC":
        dA = dA.T  # Transpose dA if it comes from a fully connected layer
    
    dA_prev = np.zeros_like(A_prev)  # Initialize dA_prev with zeros
    
    # If the mode is "max"
    if mode == "max":
        for i in range(m):
            a_prev = A_prev[i]
            for h in range(n_H_prev):  # Use n_H_prev instead of n_H
                for w in range(n_W_prev):  # Use n_W_prev instead of n_W
                    for c in range(n_C_prev):
                        vert_start = h * stride
                        vert_end = vert_start + f
                        horiz_start = w * stride
                        horiz_end = horiz_start + f
                        
                        # Compute the mask for the current window (a slice from A_prev)
                        a_prev_slice = a_prev[vert_start:vert_end, horiz_start:horiz_end, c]
                        mask = (a_prev_slice == np.max(a_prev_slice))
                        
                        # Update gradients for the previous layer
                        if layer == "Conv":
                            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, h, w, c]
                        elif layer == "FC":
                            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, c]
    
    # If the mode is "average"
    elif mode == "average":
        for i in range(m):
            for h in range(n_H_prev):  # Use n_H_prev instead of n_H
                for w in range(n_W_prev):  # Use n_W_prev instead of n_W
                    for c in range(n_C_prev):
                        vert_start = h * stride
                        vert_end = vert_start + f
                        horiz_start = w * stride
                        horiz_end = horiz_start + f
                        
                        # Compute the average value
                        da = dA[i, h, w, c]
                        avg = da / (f * f)
                        
                        # Update gradients for the previous layer
                        if layer == "Conv":
                            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += np.ones((f, f)) * avg
                        elif layer == "FC":
                            dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += np.ones((f, f)) * avg
    
    return dA_prev



"""

def pool_backward(dA, cache, mode):
 
    (A_prev, hparameters) = cache
    stride = hparameters["stride"]
    f = hparameters["f_pool"]

    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape
    m, n_H, n_W, n_C = dA.shape

    dA_prev = np.zeros(A_prev.shape)

    for i in range(m):
        a_prev = A_prev[i]
        for h in range(n_H):
            vert_start = h * stride
            vert_end = vert_start + f
            for w in range(n_W):
                horiz_start = w * stride
                horiz_end = horiz_start + f
                for c in range(n_C):
                    if mode == "max":
                        a_prev_slice = a_prev[vert_start:vert_end, horiz_start:horiz_end, c]
                        mask = (a_prev_slice == np.max(a_prev_slice))
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, h, w, c]
                    elif mode == "average":
                        da = dA[i, h, w, c]
                        shape = (f, f)
                        avg = da / (f * f)
                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += np.ones(shape) * avg

    return dA_prev
"""

'\n\ndef pool_backward(dA, cache, mode):\n \n    (A_prev, hparameters) = cache\n    stride = hparameters["stride"]\n    f = hparameters["f_pool"]\n\n    m, n_H_prev, n_W_prev, n_C_prev = A_prev.shape\n    m, n_H, n_W, n_C = dA.shape\n\n    dA_prev = np.zeros(A_prev.shape)\n\n    for i in range(m):\n        a_prev = A_prev[i]\n        for h in range(n_H):\n            vert_start = h * stride\n            vert_end = vert_start + f\n            for w in range(n_W):\n                horiz_start = w * stride\n                horiz_end = horiz_start + f\n                for c in range(n_C):\n                    if mode == "max":\n                        a_prev_slice = a_prev[vert_start:vert_end, horiz_start:horiz_end, c]\n                        mask = (a_prev_slice == np.max(a_prev_slice))\n                        dA_prev[i, vert_start:vert_end, horiz_start:horiz_end, c] += mask * dA[i, h, w, c]\n                    elif mode == "average":\n                        da = dA[i, h, w, c]\n     

In [432]:
def fully_connected_backward(dA, cache, layer):

        (linear_cache, activation_cache) = cache
        (A_prev, W, b) = linear_cache
        m = A_prev.shape[1]
    
        if layer == "Output":
            
            dZ = backward_softmax(dAL=dA, Z=activation_cache)
            dW = (1. / m) * np.dot(dZ, A_prev.T)
            db = (1. / m) * np.sum(dZ, axis=1, keepdims=True)
            dA_prev = np.dot(W.T, dZ)
            
        elif layer == "Hidden":
            
            dZ = backward_relu(dA=dA, Z=activation_cache)
            dW = (1. / m) * np.dot(dZ, A_prev.T)
            db = (1. / m) * np.sum(dZ, axis=1, keepdims=True)
            dA_prev = np.dot(W.T, dZ)

        return dA_prev, dW, db

In [433]:
def backward_propagation(AL, Y, caches, layers, pool_mode):

    Y = Y.reshape(AL.shape)
    L = len(layers)
    gradients = {}
    
    dAL = - (np.divide(Y, AL + 1e-15) - np.divide(1 - Y, 1 - AL + 1e-15))
    
    dA_prev, dW, db = fully_connected_backward(
        dA=dAL, 
        cache=caches[-1], 
        layer="Output"
    )
    
    gradients[f"dW{L}"] = dW
    gradients[f"db{L}"] = db
    gradients[f"dA{L-1}"] = dA_prev
    
    for l in reversed(range(L-1)):

        if layers[l] == "FC_hidden":
            
            dA_prev, dW, db = fully_connected_backward(
                dA=gradients[f"dA{l+1}"], 
                cache=caches[l], 
                layer="Hidden"
            )
            
            gradients[f"dW{l+1}"] = dW
            gradients[f"db{l+1}"] = db
            gradients[f"dA{l}"] = dA_prev
            
        elif layers[l] == "Conv" and layers[l+1]== "FC_hidden":
            
            (conv_cache, pool_cache) = caches[l]
            dA_prev = pool_backward(
                dA=gradients[f"dA{l+1}"], 
                cache=pool_cache, 
                mode=pool_mode,
                layer="FC"
            )
            dA_prev, dW, db = convolution_backward(
                dA=dA_prev, 
                cache=conv_cache
            )
            
            gradients[f"dW{l+1}"] = dW
            gradients[f"db{l+1}"] = db
            gradients[f"dA{l}"] = dA_prev
            
        elif layers[l] == "Conv" and layers[l+1]== "Conv":
            
            (conv_cache, pool_cache) = caches[l]
            dA_prev = pool_backward(
                dA=gradients[f"dA{l+1}"], 
                cache=pool_cache, 
                mode=pool_mode,
                layer="Conv"
            )
        
            dA_prev, dW, db = convolution_backward(
                dZ=dA_prev, 
                cache=conv_cache
            )
            
            gradients[f"dW{l+1}"] = dW
            gradients[f"db{l+1}"] = db
            gradients[f"dA{l}"] = dA_prev
            
    return gradients   

In [434]:
def compute_first_momentum(parameters, gradients, V, beta, t):
    
    L = len(parameters) // 2

    for l in range(1, L + 1):
        
        V[f"vdW{l}"] = (beta * V[f"vdW{l}"]) + ((1 - beta) * gradients[f"dW{l}"])
        V[f"vdb{l}"] = (beta * V[f"vdb{l}"]) + ((1 - beta) * gradients[f"db{l}"])

        V[f"vdW{l}"] = V[f"vdW{l}"] / (1 - np.power(beta, t))
        V[f"vdb{l}"] = V[f"vdb{l}"] / (1 - np.power(beta, t))

    return V

In [435]:
def compute_second_momentum(parameters, gradients, S, beta, t):
    
    L = len(parameters) // 2

    for l in range(1, L + 1):
        
        S[f"sdW{l}"] = (beta * S[f"sdW{l}"]) + ((1 - beta) * np.power(gradients[f"dW{l}"], 2))
        S[f"sdb{l}"] = (beta * S[f"sdb{l}"]) + ((1 - beta) * np.power(gradients[f"db{l}"], 2))

        S[f"sdW{l}"] = S[f"sdW{l}"] / (1 - np.power(beta, t))
        S[f"sdb{l}"] = S[f"sdb{l}"] / (1 - np.power(beta, t))

        return S

In [436]:
def update_parameters(parameters, V, S, learning_rate, epsilon):

        L = len(parameters) // 2

        for l in range(1, L+1):
            
            parameters[f"W{l}"] -= learning_rate * (V[f"vdW{l}"] / np.sqrt(S[f"sdW{l}"] + epsilon))
            parameters[f"b{l}"] -= learning_rate * (V[f"vdb{l}"] / np.sqrt(S[f"sdb{l}"] + epsilon))

        return parameters

In [437]:
def update_learning_rate(init_learning_rate, epoch_num, decay_rate, time_interval):

        learning_rate = (1 * init_learning_rate) / (1 + decay_rate * (np.floor(epoch_num / time_interval)))

        return learning_rate

In [438]:
def predict(X, parameters, hyperparameters, layers, pool_mode):

        A, _ = forward_propagation(X=X, parameters=parameters, hyperparameters=hyperparameters, layers=layers, pool_mode=pool_mode)

        return A

In [439]:
def accuracy(A, Y):
    
    assert (A.shape == Y.shape)

    predicted_labels = np.argmax(A, axis=0)
    true_labels = np.argmax(Y, axis=0)

    accuracy = np.mean(predicted_labels == true_labels)

    return accuracy

In [440]:
def random_mini_batches(X, Y, mini_batch_size, seed):
    
    np.random.seed(seed)
    m = X.shape[0]  
    num_classes = Y.shape[0] 

    permutation = np.random.permutation(m)
    shuffled_X = X[permutation]
    shuffled_Y = Y[:, permutation]

    num_complete_minibatches = m // mini_batch_size
    mini_batches = []

    for k in range(num_complete_minibatches):
        start_idx = k * mini_batch_size
        end_idx = (k + 1) * mini_batch_size
        mini_batch_X = shuffled_X[start_idx:end_idx]
        mini_batch_Y = shuffled_Y[:, start_idx:end_idx]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    if m % mini_batch_size != 0:
        start_idx = num_complete_minibatches * mini_batch_size
        mini_batch_X = shuffled_X[start_idx:]
        mini_batch_Y = shuffled_Y[:, start_idx:]
        mini_batches.append((mini_batch_X, mini_batch_Y))

    return mini_batches

In [441]:
def train(
    X_train, Y_train, X_test, Y_test, num_fc_units, conv_output, 
    layers=["Conv", "Conv", "FC_hidden", "FC_output"], f_size=3, 
    num_channels=3, num_filters=3, num_epochs=100, learning_rate=1e-3, beta1=0.9, 
    beta2=0.999, seed=1, minibatch_size=16, stride=1, pad=1, f_pool=2,
    pool_mode="max", epsilon=1e-8, decay_rate=0.3, time_interval=50,
    pad_mode="constant", constant_values=(0, 0)
):
    
    costs = []

    Y_train = onehot_encoding(Y=Y_train)
    Y_test = onehot_encoding(Y=Y_test)
    
    layer_units = layer_sizes(
        num_fc_units=num_fc_units, 
        layers=layers
    )
    (parameters, V, S) = initialize_parameters(
        layers=layers, 
        f_size=f_size, 
        num_channels=num_channels,
        num_filters =num_filters,
        layer_units=layer_units, 
        conv_output=conv_output
    )
    
    init_learning_rate = learning_rate
    epoch_num = 0
    m = X_train.shape[0]
    t = 0
    
    hyperparameters = {
        "stride": stride, 
        "pad": pad, 
        "f_pool": f_pool,
        "pad_mode": pad_mode,
        "constant_values": constant_values
    }
    
    for i in range(num_epochs):
        
        seed = seed + 1
        mini_batches = random_mini_batches(
            X=X_train,
            Y=Y_train,
            mini_batch_size=minibatch_size,
            seed=seed
        )
            
        cost_total = 0
        for minibatch in mini_batches:
            
            (X_minibatch, Y_minibatch) = minibatch
            A_minibatch, minibatch_caches = forward_propagation(
                X=X_minibatch,
                parameters=parameters,
                hyperparameters=hyperparameters,
                layers=layers,
                pool_mode=pool_mode
            )
            cost_total += compute_cost(
                A=A_minibatch,
                Y=Y_minibatch,
                dataset="train"
            )
            
            gradients = backward_propagation(
                AL=A_minibatch,
                Y=Y_minibatch,
                caches=minibatch_caches,
                layers=layers,
                pool_mode=pool_mode
            )
            t += 1
            V = compute_first_momentum(
                parameters=parameters,
                gradients=gradients,
                V=V,
                beta=beta1,
                t=t
            )
            S = compute_second_momentum(
                parameters=parameters,
                gradients=gradients,
                S=S,
                beta=beta2,
                t=t
            )
            parameters = update_parameters(
                parameters=parameters,
                V=V,
                S=S,
                learning_rate=learning_rate,
                epsilon=epsilon
            )

        test_mini_batches = random_mini_batches(
            X=X_test,
            Y=Y_test,
            mini_batch_size=minibatch_size,
            seed=seed
        )
        test_cost_total = 0.0
        
        for minibatch in test_mini_batches:
            
            (X_minibatch, Y_minibatch) = minibatch
            A_minibatch, minibatch_caches = forward_propagation(
                    X=X_minibatch,
                    parameters=parameters,
                    hyperparameters=hyperparameters,
                    layers=layers,
                    pool_mode=pool_mode
            )
            test_cost_total += compute_cost(
                A=A_minibatch,
                Y=Y_minibatch,
                dataset="test"
            )

        costs.append(cost_total / m)
        epoch_num += 1
        learning_rate = update_learning_rate(
            init_learning_rate=init_learning_rate,
            epoch_num=epoch_num,
            decay_rate=decay_rate,
            time_interval=time_interval
        )
        print(f"Epoch {epoch_num}")
        
    return parameters, costs

In [442]:
def load_dataset(train_path, test_path):
    
    train_data = h5py.File(train_path, "r")
    test_data = h5py.File(test_path, "r")
    X_train = np.array(train_data["train_set_x"])
    Y_train = np.array(train_data["train_set_y"])
    X_test = np.array(test_data["test_set_x"])
    Y_test = np.array(test_data["test_set_y"])
    classes = np.array(test_data["list_classes"])
    
    return X_train, Y_train, X_test, Y_test, classes

In [443]:
def preprocess_data(X):
    
    X = X / 255.
    X = np.transpose(X, (0, 1, 2, 3))
    
    return X

In [444]:
X_train, Y_train, X_test, Y_test, classes = load_dataset("/home/samani/Documents/projects/deep-learning/data/train_signs.h5", "/home/samani/Documents/projects/deep-learning/data/test_signs.h5")
print(X_train.shape)
# Preprocess the data
X_train = preprocess_data(X_train)
X_test = preprocess_data(X_test)

(1080, 64, 64, 3)


In [445]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(1080, 64, 64, 3)
(120, 64, 64, 3)
(1080,)
(120,)


In [446]:
params, costs = train(
    X_train=X_train, Y_train=Y_train, X_test=X_test, 
    Y_test=Y_test, num_fc_units=["_", "_", 20, 6], conv_output=10800, 
    layers=["Conv", "Conv", "FC_hidden", "FC_output"], f_size=3, 
    num_channels=3, num_epochs=10, learning_rate=1e-3, beta1=0.9, 
    beta2=0.999, seed=1, minibatch_size=16, stride=1, pad=1, f_pool=3,
    pool_mode="max", epsilon=1e-8, decay_rate=0.3, time_interval=50,
    pad_mode="constant", constant_values=(0, 0)
)

IndexError: index 62 is out of bounds for axis 2 with size 62